# TrustOCT — Training Notebook 2: Proposed Attention Model
### Model: `msf_cbam_resnet50` (Proposed Winner)


In [ ]:
import os
if not os.path.exists('src'):
    os.system('git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git repo_temp')
    os.system('cp -r repo_temp/* . && rm -rf repo_temp')
try:
    import albumentations, ptflops
except ImportError:
    os.system('pip install -r requirements.txt')


In [ ]:
import os, sys, time, cv2, numpy as np, pandas as pd, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f'Drive mount note: {e}')


In [ ]:
# Section 4 — Kermany Dataset Mapping & Patient-Level Split
from src.datasets.dataset import auto_detect_columns, patient_level_split, RetinalDataset
from src.preprocessing.standardizer import RetinalPipelineTransform
from torch.utils.data import DataLoader
import pandas as pd, os

csv_path = 'kermany_dataset_mapping.csv'
root_oct = None
for cand in ['/content/Kermany', '/content/OCT2017', '/content/Kermany/OCT2017', '/content/Kermany/OCT2017 ']:
    if os.path.exists(cand):
        root_oct = cand
        break

if root_oct and (not os.path.exists(csv_path) or os.path.getsize(csv_path) < 100):
    print(f'🔍 Scanning dataset directory across train/val/test folders at {root_oct}...')
    records = []
    class_map = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
    for root, dirs, files_list in os.walk(root_oct):
        for f in files_list:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent = os.path.basename(root).lower()
                lbl = class_map.get(parent, -1)
                if lbl != -1:
                    base = os.path.splitext(f)[0]
                    parts = base.split('-')
                    pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                    records.append({'image_path': os.path.join(root, f), 'label': int(lbl), 'patient_id': pt_id})
    if records:
        df_all = pd.DataFrame(records)
        df_all.to_csv(csv_path, index=False)
        print(f'✅ Generated kermany_dataset_mapping.csv with {len(df_all)} total images!')

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    train_df, val_df, test_df = patient_level_split(df)
    print(f'✅ Patient-Level Split -> Train: {len(train_df)} images, Val: {len(val_df)} images, Test: {len(test_df)} images')
    transform = RetinalPipelineTransform(is_training=False)
    test_dataset = RetinalDataset(test_df, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
epochs = 30
lr = 1e-4
batch_size = 32


### 3. Train `msf_cbam_resnet50` Proposed Model


In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_cbam_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))


In [ ]:
import shutil
shared_outputs = '/content/drive/MyDrive/TrustOCT_Results/outputs'
os.makedirs(shared_outputs, exist_ok=True)
if os.path.exists('outputs/msf_cbam_resnet50'):
    dst = os.path.join(shared_outputs, 'msf_cbam_resnet50')
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree('outputs/msf_cbam_resnet50', dst)
    print('Synced msf_cbam_resnet50 results to Drive.')
